In [3]:
:dep smartcore = { version = "0.3.2", features = ["ndarray-bindings", "serde"] }
:dep bincode = "1.3"
:dep csv = "1.1"
:dep ndarray = { version = "0.16", features = ["rayon"] }
:dep linfa = { version = "0.8.1" }
:dep rand = "0.8"
:dep rand_chacha = "0.3"
:dep plotters = "0.3"
:dep linfa-preprocessing =  "0.8.1" 
:dep linfa-elasticnet = { version =  "0.8.1" }
:dep polars = { version = "0.53", features = ["ndarray"] }
:dep polars-core = "0.53"
:dep linfa-reduction =  "0.8.1" 
:dep image = "0.24"
:opt 3

Optimization: 3


In [13]:
use linfa::prelude::*;
use linfa_reduction::Pca;
use plotters::prelude::*;
use ndarray::Array2;
use std::path::Path as StdPath;
use linfa_preprocessing::linear_scaling::LinearScaler;
use plotters::prelude::*;
use std::path::Path as StdPath;
use image::GenericImageView;
use polars_core::prelude::*;
use ndarray::Dim;
use ndarray::ArrayBase;
use ndarray::OwnedRepr;
use polars::prelude::CsvReadOptions;
use polars::prelude::SerReader;

In [23]:
let mut df = CsvReadOptions::default()
        .with_has_header(false)
        .try_into_reader_with_file_path(Some("FlagsData/flag.data".into())).unwrap()
        .finish().unwrap();
let color_cols = ["column_18", "column_29", "column_30"];
let mut color_map: std::collections::HashMap<String, f64> = std::collections::HashMap::new();
let mut counter = 0.0;
for col_name in color_cols {
    let series = df.column(col_name).unwrap().as_materialized_series();
    for val in series.iter() {
        let color_str = val.to_string().replace("\"", "");
        let _ = color_map.entry(color_str).or_insert_with(|| {
            let current = counter;
            counter += 1.0;
            current
        });
    }
}
for col_name in color_cols {
    let series = df.column(col_name).unwrap().as_materialized_series();
    let values: Vec<f64> = series.iter().map(|val| {
        let color_str = val.to_string().replace("\"", "");
        *color_map.get(&color_str).unwrap_or(&0.0)
    }).collect();
    let new_col = Series::new(col_name.into(), values);
    let _ = df.replace(col_name, new_col.into_column()).unwrap();
}

()

In [24]:
let feature_cols: Vec<String> = (2..=30).map(|i| format!("column_{}", i)).collect();
let shape = (df.height(), feature_cols.len());
let x_vec: Vec<f64> = df.select(&feature_cols)
    .unwrap()
    .to_ndarray::<Float64Type>(IndexOrder::C)
    .unwrap()
    .into_iter()
    .collect();
let x_final = ndarray::Array2::from_shape_vec(shape, x_vec).unwrap();
let y_final = ndarray::Array1::from_iter(0..shape.0);
let dataset  = Dataset::new(x_final, y_final);
let scaler: LinearScaler<f64> = LinearScaler::standard().fit(&dataset).expect("errr");
let dataset = scaler.transform(dataset);
let pca: Pca<f64> = Pca::params(2).fit(&dataset).unwrap();
let records = pca.predict(&dataset);

In [20]:
let countries: Vec<String> = df.column("column_1").clone()
    .unwrap()
    .str()          
    .unwrap()
    .into_iter()     
    .map(|opt_val| opt_val.unwrap_or("unknown").to_string()) 
    .collect();
let root = BitMapBackend::new("pca_standart.png", (4000, 4000)).into_drawing_area();
root.fill(&WHITE).unwrap();
let mut chart = ChartBuilder::on(&root)
    .caption("PCA", ("sans-serif", 40))
    .margin(20)
    .build_cartesian_2d(-1000.0..1000.0, -1000.0..1000.0).unwrap();
chart.configure_mesh().draw().unwrap();
for id in 0..countries.len(){
    for entry in std::fs::read_dir("Flags/").unwrap(){
        let file = entry.unwrap();
        let name = file.file_name().to_string_lossy().into_owned();
        if name.contains(&countries[id]){
            let img = image::open(file.path()).unwrap();
            let (width, height) = img.dimensions();
            let row = records.row(id);
            let rgb_img = img.to_rgb8();
            let plt_img = BitMapElement::<(f64, f64), plotters::backend::RGBPixel>::with_ref(
                (row[0]*100.0, row[1]*100.0),
                rgb_img.dimensions(),
                rgb_img.as_raw()
            ).expect("pepes");
            
            chart.draw_series(std::iter::once(plt_img)).unwrap();
            break;
        }
    }
}
root.present().unwrap();
let img = image::open("pca_standart.png").expect("nfs");
let scaled = img.resize(1000, 1000, image::imageops::FilterType::Lanczos3);
scaled.save("pca_standart.png").expect("peps");

In [25]:
let pca: Pca<f64> = Pca::params(2).whiten(true).fit(&dataset).unwrap();
let records = pca.predict(&dataset);
let countries: Vec<String> = df.column("column_1").clone()
    .unwrap()
    .str()          
    .unwrap()
    .into_iter()     
    .map(|opt_val| opt_val.unwrap_or("unknown").to_string()) 
    .collect();
let root = BitMapBackend::new("pca_whiten.png", (4000, 4000)).into_drawing_area();
root.fill(&WHITE).unwrap();
let mut chart = ChartBuilder::on(&root)
    .caption("PCA", ("sans-serif", 40))
    .margin(20)
    .build_cartesian_2d(-1000.0..1000.0, -1000.0..1000.0).unwrap();
chart.configure_mesh().draw().unwrap();
for id in 0..countries.len(){
    for entry in std::fs::read_dir("Flags/").unwrap(){
        let file = entry.unwrap();
        let name = file.file_name().to_string_lossy().into_owned();
        if name.contains(&countries[id]){
            let img = image::open(file.path()).unwrap();
            let (width, height) = img.dimensions();
            let row = records.row(id);
            let rgb_img = img.to_rgb8();
            let plt_img = BitMapElement::<(f64, f64), plotters::backend::RGBPixel>::with_ref(
                (row[0]*100.0, row[1]*100.0),
                rgb_img.dimensions(),
                rgb_img.as_raw()
            ).expect("pepes");
            
            chart.draw_series(std::iter::once(plt_img)).unwrap();
            break;
        }
    }
}
root.present().unwrap();
let img = image::open("pca_whiten.png").expect("nfs");
let scaled = img.resize(1000, 1000, image::imageops::FilterType::Lanczos3);
scaled.save("pca_whiten.png").expect("peps");